In [151]:
import pandas as pd
import numpy as np
import ast
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [152]:
EJEMPLO_USUARIO = "1000rpm"
VALORACION_USUARIO_UMBRAL = 5

In [153]:
mini_reviews = pd.read_csv("../data/bgg-26m-reviews-mini.csv")
info_juegos_optimizado = pd.read_csv("../data/info_juegos_optimizado.csv")

In [154]:
mini_reviews.head(11)

,user,rating,comment,ID
0,Calo,10,"Just a great, fun game",10630
1,ufo5440,10,"I finally have the game and played it, and you...",10630
2,dogsbody40,10,Great Fast WWII Fun,10630
3,zapator,10,"Very cool gameplay, simple, fast and very tact...",10630
4,Bloodybucket,10,Great fun! Will see a lot of play. Repeated ...,10630
5,pound,10,"I love it: the new terrain effects, the close ...",10630
6,ingensorg,10,Played a few times and love the game. Relative...,10630
7,Midian2000,10,"Sheer fun. Easy to teach, learn, and play. A...",10630
8,edralla,10,Great game! Easy to play right out of the box...,10630
9,PAYDIRT,10,Brings back the days of playing in the dirt in...,10630


In [155]:
info_juegos_optimizado.head()

,id,primary,yearpublished,maxplayers,playingtime,boardgamecategory,average
0,10630,memoir_'44,2004,8.0,60.0,"['Miniatures', 'Wargame', 'World War II']",7.56
1,17133,railways_of_the_world,2005,6.0,120.0,"['Trains', 'Transportation', 'Video Game Theme']",7.69
2,120,hoity_toity,1990,6.0,45.0,['Bluffing'],6.52
3,6351,gulo_gulo,2003,6.0,20.0,"['Action / Dexterity', 'Animals', ""Children's ...",6.83
4,19948,rum_&_pirates,2006,5.0,60.0,"['Dice', 'Miniatures', 'Pirates']",6.43


In [156]:
rated_games = mini_reviews.groupby("ID").agg({"rating": "mean", "user": "count"}).reset_index()
rated_games.sort_values("rating", ascending=False)

,ID,rating,user
3,17133,7.585741,2665
2,10630,7.423547,6056
1,6351,6.933921,1135
0,120,6.355586,1468
4,19948,6.247148,526


In [157]:
games_matrix = mini_reviews.pivot_table(index='user', columns='ID', values='rating')

# Se rellenan los nulos con ceros para cálculos posteriores.
# games_matrix.fillna(0) 
games_matrix.head()

ID,120,6351,10630,17133,19948
user,,,,,
-Johnny-,NaN,NaN,NaN,7.0,NaN
-Morphling-,NaN,NaN,7.0,NaN,NaN
0v3rl0r17,NaN,NaN,8.0,NaN,NaN
1 Family Meeple,NaN,NaN,3.0,7.0,NaN
1000days,NaN,NaN,8.0,NaN,NaN


In [158]:
user_ratings = games_matrix.loc[EJEMPLO_USUARIO]
user_ratings = user_ratings.dropna()
user_ratings.head()

ID
6351     7.0
10630    8.0
Name: 1000rpm, dtype: float64

In [159]:
juegos_valorados = user_ratings[user_ratings >= VALORACION_USUARIO_UMBRAL]
df_juegos_valorados = juegos_valorados.reset_index()
df_juegos_valorados.columns = ['ID', 'rating']
df_juegos_valorados.head()

,ID,rating
0,6351,7.0
1,10630,8.0


In [160]:
similitud_ponderada = pd.Series(np.zeros(games_matrix.shape[1]), index=games_matrix.columns)
similitud_ponderada.head()

ID
120      0.0
6351     0.0
10630    0.0
17133    0.0
19948    0.0
dtype: float64

In [ ]:
for game_id in juegos_valorados.keys():
    # Obtiene la correlación del juego valorado del usuario objetivo con el resto de juegos de games_matrix y devolviendo una serie
    # La correlación va desde -1 a 1
    corr_series = games_matrix.corrwith(games_matrix[game_id])
    corr_series = corr_series + 1
    # Reemplazamos NaN por 0 en la correlación
    corr_series = corr_series.fillna(0)
    # Se suma esa correlación a cada juego de similitud_ponderada
    similitud_ponderada = similitud_ponderada.add(corr_series, fill_value=0)
    

similitud_ponderada = similitud_ponderada.astype(np.float16)

In [162]:
len(juegos_valorados)

2

In [163]:
peso_total = len(juegos_valorados) * 2
if peso_total != 0:
    similitud_ponderada = similitud_ponderada / peso_total
    
print(similitud_ponderada.sort_values(ascending=False).head())
len(similitud_ponderada)

ID
6351     0.804688
10630    0.804688
19948    0.666504
17133    0.608887
120      0.599121
dtype: float16


5

In [164]:
df_similitud_ponderada = similitud_ponderada.reset_index()
df_similitud_ponderada.columns = ['ID', 'ponderacion']

In [165]:
sia = SentimentIntensityAnalyzer()
mini_reviews['sentiment'] = mini_reviews['comment'].apply(
    lambda comment: (sia.polarity_scores(comment)['compound'] + 1) if isinstance(comment, str) else 0)

In [166]:
df_avg_sentiment = mini_reviews.groupby('ID')['sentiment'].mean().reset_index().rename(columns={'sentiment': 'avg_sentiment'})
df_avg_sentiment['avg_sentiment'] = df_avg_sentiment['avg_sentiment'] * 0.5
df_avg_sentiment.sort_values(by='avg_sentiment', ascending=False).head()

,ID,avg_sentiment
1,6351,0.803770
3,17133,0.763202
4,19948,0.739170
2,10630,0.725363
0,120,0.692735


In [ ]:
# Elimina aquellos juegos que el usuario objetivo ya ha valorado
user_ratings = user_ratings.index.tolist()
df_similitud_ponderada = df_similitud_ponderada[~df_similitud_ponderada['ID'].isin(user_ratings)]
df_avg_sentiment = df_avg_sentiment[~df_avg_sentiment['ID'].isin(user_ratings)]
print(len(df_similitud_ponderada))
print(len(df_avg_sentiment))

3
3


In [168]:
df_similitud_ponderada = df_similitud_ponderada.sort_values(ascending=False, by='ponderacion')
df_similitud_ponderada.head()

,ID,ponderacion
4,19948,0.666504
3,17133,0.608887
0,120,0.599121


In [169]:
df_analizado = df_similitud_ponderada.merge(info_juegos_optimizado, left_on='ID', right_on='id', how='left')
df_analizado = df_analizado.drop(['id'], axis=1)
df_analizado = df_analizado.merge(df_avg_sentiment, left_on='ID', right_on='ID', how='left')
df_analizado.head()

,ID,ponderacion,primary,yearpublished,maxplayers,playingtime,boardgamecategory,average,avg_sentiment
0,19948,0.666504,rum_&_pirates,2006,5.0,60.0,"['Dice', 'Miniatures', 'Pirates']",6.43,0.739170
1,17133,0.608887,railways_of_the_world,2005,6.0,120.0,"['Trains', 'Transportation', 'Video Game Theme']",7.69,0.763202
2,120,0.599121,hoity_toity,1990,6.0,45.0,['Bluffing'],6.52,0.692735


In [170]:
# La información completa de aquellos juegos valorados por el usuario objetivo
juegos_valorados_info = info_juegos_optimizado[info_juegos_optimizado['id'].isin(df_juegos_valorados['ID'])]


# Filtrado de categorías jugadas por el usuario objetivo
game_category = juegos_valorados_info['boardgamecategory'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

FILTRO_CATEGORIA = {categoria for lista in game_category for categoria in lista}
juegos_valorados_info.count()

id                   2
primary              2
yearpublished        2
maxplayers           2
playingtime          2
boardgamecategory    2
average              2
dtype: int64

In [173]:
print(FILTRO_CATEGORIA)

{'World War II', 'Humor', 'Miniatures', "Children's Game", 'Action / Dexterity', 'Animals', 'Wargame'}


In [171]:
# Construir la máscara inicial con False para todas las filas de recommender_results
mask = pd.Series(False, index=df_analizado.index)

# Para cada categoría de las valoradas por el usuario, actualiza la máscara
for categoria in FILTRO_CATEGORIA:
    mask |= df_analizado['boardgamecategory'].fillna("").str.contains(categoria, case=False, regex=True)

df_analizado = df_analizado[mask]

df_analizado.head()

,ID,ponderacion,primary,yearpublished,maxplayers,playingtime,boardgamecategory,average,avg_sentiment
0,19948,0.666504,rum_&_pirates,2006,5.0,60.0,"['Dice', 'Miniatures', 'Pirates']",6.43,0.73917


In [172]:
df_analizado['recomendacion_calculada'] = (df_analizado['ponderacion']) * 0.8 + (df_analizado['avg_sentiment'] * 0.2)
df_analizado = df_analizado.nlargest(5, 'recomendacion_calculada')
print(f"Recomendaciones para el usuario {EJEMPLO_USUARIO}")
print(df_analizado[['ID', 'primary', 'recomendacion_calculada']])

Recomendaciones para el usuario 1000rpm
      ID        primary  recomendacion_calculada
0  19948  rum_&_pirates                 0.681037
